# Notebook 2: Transform data records into dataframes

Import the Python library dependencies.

In [1]:
import pathlib

import polars as pl
import watermark

Run a "watermark" to show which library versions are used in this notebook's runtime environment.

In [2]:
%load_ext watermark
%watermark
%watermark --iversions

Last updated: 2025-11-12T10:48:10.763408-08:00

Python implementation: CPython
Python version       : 3.13.8
IPython version      : 9.1.0

Compiler    : Clang 17.0.0 (clang-1700.0.13.3)
OS          : Darwin
Release     : 24.6.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

watermark: 2.5.0
polars   : 1.29.0



## Parse the OpenSanctions dataset

In [3]:
df_src: pl.DataFrame = pl.read_ndjson("data/open-sanctions.json")

# normalize IRIs for connecting later
df_src = df_src.replace_column(
    1,
    pl.Series(
        "id",
        map(lambda s: "sz:ds_open-sanctions_" + s, df_src["RECORD_ID"])
    ),
)

df_src = df_src.replace_column(
    2,
    pl.Series(
        "class",
        map(lambda s: "sz:" + s.capitalize(), df_src["RECORD_TYPE"])
    ),
)

In [4]:
df_src

DATA_SOURCE,id,class,LAST_CHANGE,NAMES,GENDER,RISKS,ADDRESSES,DATES,COUNTRIES,IDENTIFIERS,SOURCE_LINKS,RELATIONSHIPS,URL,CONTACTS
str,str,str,str,list[struct[3]],str,list[struct[1]],list[struct[7]],list[struct[2]],list[struct[3]],list[struct[9]],list[struct[1]],list[struct[5]],str,list[struct[1]]
"""OPEN-SANCTIONS""","""sz:ds_open-sanctions_NK-25vyVF…","""sz:Person""","""2024-07-30T16:41:14""","[{""PRIMARY"",null,""Abassin BADSHAH""}]",null,"[{""corp.disqual""}]","[{""31 Quernmore Close, Bromley, Kent, United Kingdom, BR1 4EL"",null,null,null,null,null,null}]","[{null,""1985-05-12""}]","[{null,""gb"",null}]","[{null,null,null,null,null,""OPEN-SANCTIONS"",""NK-25vyVFzt8vdJGgAXMRTwTJ"",null,null}]","[{""https://find-and-update.company-information.service.gov.uk/disqualified-officers/natural/mGquuTbmESWiRmHJPz1ObUwfDgk""}]","[{""Directorship"",""OPEN-SANCTIONS"",""NK-SKAADAiqiZ78JsJjeg72Te"",null,null}, {""Directorship"",""OPEN-SANCTIONS"",""NK-3p3mmVWmjwVtTfKchz4kNE"",null,null}]","""https://www.opensanctions.org/…",null
"""OPEN-SANCTIONS""","""sz:ds_open-sanctions_NK-3p3mmV…","""sz:Organization""","""2025-01-07T00:33:03""","[{""PRIMARY"",""LMAR (GB) LTD"",null}]",null,null,"[{""31 Quernmore Close, Bromley, Kent, United Kingdom, BR1 4EL"",null,null,null,null,null,""BUSINESS""}]",null,"[{""gb"",null,null}]","[{null,null,null,null,null,""OPEN-SANCTIONS"",""NK-3p3mmVWmjwVtTfKchz4kNE"",null,null}]",null,"[{null,null,null,""OPEN-SANCTIONS"",""NK-3p3mmVWmjwVtTfKchz4kNE""}]","""https://www.opensanctions.org/…",null
"""OPEN-SANCTIONS""","""sz:ds_open-sanctions_NK-auyPsL…","""sz:Organization""","""2024-03-03T19:51:29""","[{""PRIMARY"",""WANDLE HOLDINGS LIMITED"",null}]",null,"[{""sanction.linked""}]","[{""DEANA BEACH APTS, BLOCK A, Flat 212, Προμαχών Ελευθερίας, 33, 'Αγιος Αθανάσιος, 4103, Λεμεσός, Κύπρος"",null,null,null,null,null,""BUSINESS""}]","[{""2006-12-08"",null}]","[{""cy"",null,null}]","[{""C188266"",null,null,null,null,null,null,null,null}, {""HE188266"",null,null,null,null,null,null,null,null}, {null,null,null,null,null,""OPEN-SANCTIONS"",""NK-auyPsLrBzRoxjCRWgjBvas"",null,null}]",null,"[{null,null,null,""OPEN-SANCTIONS"",""NK-auyPsLrBzRoxjCRWgjBvas""}]","""https://opensanctions.org/enti…",null
"""OPEN-SANCTIONS""","""sz:ds_open-sanctions_NK-cf4Q3K…","""sz:Organization""","""2024-03-03T08:46:31""","[{""PRIMARY"",""POLYUS GOLD INTERNATIONAL LIMITED"",null}, {""ALIAS"",""KAZAKHGOLD GROUP LIMITED"",null}]",null,"[{""sanction.linked""}]","[{""3RD FLOOR CHARTER PLACE 23-27 SEATON PALCE, ST HELIER JE4 0WH"",null,null,null,null,null,""BUSINESS""}]","[{""2007-09-12"",null}]","[{""gb"",null,null}, {""je"",null,null}]","[{""FC027918"",null,null,null,null,null,null,null,null}, {null,null,null,null,null,""OPEN-SANCTIONS"",""NK-cf4Q3KcmUnQbt8Cy7iTtwK"",null,null}]","[{""http://business.data.gov.uk/id/company/FC027918""}]","[{null,null,null,""OPEN-SANCTIONS"",""NK-cf4Q3KcmUnQbt8Cy7iTtwK""}]","""https://opensanctions.org/enti…",null
"""OPEN-SANCTIONS""","""sz:ds_open-sanctions_NK-dNNN56…","""sz:Person""","""2025-02-10T15:38:02""","[{""PRIMARY"",null,""Firuza Nazimovna Kerimova""}, {""ALIAS"",null,""Firuza Nazimovna Khanbalaeva""}, … {""ALIAS"",null,""フィルザ・ケリモヴァ""}]","""F""","[{""role.rca""}, {""sanction""}]","[{""MOSCOW, RUS, 123430"",null,null,null,null,null,null}, {""Apt. 270, Build. 31, Pyatnitskoe Shosse, 123430 Moscow"",null,null,null,null,null,null}, … {""Apt 270, Build. 31, Pyatnitskoe Shosse, Moscow, 123430"",""Apt 270, Build. 31, Pyatnitskoe Shosse"",""Moscow"",""ru"",""123430"",null,null}]","[{null,""1967-12-22""}, {null,""1967-10-22""}]","[{null,""ru"",null}, {null,null,""ru""}]","[{null,null,""724348524"",null,null,null,null,null,null}, {""4512970434"",null,null,null,null,null,null,null,null}, … {null,null,null,null,null,""OPEN-SANCTIONS"",""NK-dNNN56A4ApVfUFvfzniLCF"",null,null}]","[{""https://sanctionssearch.ofac.treas.gov/Details.aspx?id=38277""}]","[{""Family"",""OPEN-SANCTIONS"",""Q447250"",null,null}]","""https://ww

In [5]:
# normalize names from nested attributes
df: pl.DataFrame = df_src.explode("NAMES")

df = df.with_columns(
    pl.when(pl.col("NAMES").struct.field("NAME_TYPE") == "PRIMARY_NAME_FULL")
    .then(pl.col("NAMES").struct.field("NAME_TYPE"))
    .when(pl.col("NAMES").struct.field("NAME_TYPE") == "PRIMARY")
    .then(
        pl.coalesce(
            pl.col("NAMES").struct.field("NAME_FULL"),
            pl.col("NAMES").struct.field("NAME_ORG")
        )
    )
    .otherwise(None)
    .alias("descrip")
)

# select the columns to keep
df = df.select(
        pl.col("id"),
        pl.col("descrip"),
        pl.col("class"),
        pl.col("ADDRESSES").alias("addr"),
        pl.col("URL").alias("url"),
    ).drop_nulls(subset=["descrip"]).sort("id")

# normalize addresses from nested attributes
df = df.explode("addr")

df = df.with_columns(
    pl.when(pl.col("addr").is_not_null())
    .then(
        pl.coalesce(
            pl.col("addr").struct.field("ADDR_FULL"),
        )
    )
    .otherwise(None)
    .alias("addr")
)

df

id,descrip,class,addr,url
str,str,str,str,str
"""sz:ds_open-sanctions_NK-25vyVF…","""Abassin BADSHAH""","""sz:Person""","""31 Quernmore Close, Bromley, K…","""https://www.opensanctions.org/…"
"""sz:ds_open-sanctions_NK-3p3mmV…","""LMAR (GB) LTD""","""sz:Organization""","""31 Quernmore Close, Bromley, K…","""https://www.opensanctions.org/…"
"""sz:ds_open-sanctions_NK-L2UmsZ…","""Gulnara Suleimanova KERIMOVA""","""sz:Person""","""MOSCOW, RUS, 123430""","""https://www.opensanctions.org/…"
"""sz:ds_open-sanctions_NK-L2UmsZ…","""Gulnara Suleimanova KERIMOVA""","""sz:Person""","""Apt 270, Build. 31, Pyatnitsko…","""https://www.opensanctions.org/…"
"""sz:ds_open-sanctions_NK-L2UmsZ…","""Gulnara Suleimanova KERIMOVA""","""sz:Person""","""Apt 270, Build. 31, Pyatnitsko…","""https://www.opensanctions.org/…"
…,…,…,…,…
"""sz:ds_open-sanctions_rupep-com…","""Vencher Management Limited LLC""","""sz:Organization""","""ПЕР. СТАРОМОНЕТНЫЙ, Москва""","""https://opensanctions.org/enti…"
"""sz:ds_open-sanctions_rupep-com…","""Natsionalnaia Kinoset LLC""","""sz:Organization""","""переулок Старомонетны, Москва""","""https://opensanctions.org/enti…"
"""sz:ds_open-sanctions_rupep-com…","""Zareche-Estate LLC""","""sz:Organization""","""улица Народная, Москва""","""https://opensanctions.org/enti…"


Serialize this dataframe to the `os_data.csv` CSV file.

In [6]:
df.write_csv(pathlib.Path("os_data.csv"), separator = ",")

... except that `Polars.DataFrame.unique()` appears to be flaky, so we'll use Bash commands.

In [7]:
!cat os_data.csv | sort -u -t, -k1,1 > foo.csv
!mv foo.csv os_data.csv

Also extract the risk information from Open Sanctions, which will populate nodes in a different table.

In [8]:
df: pl.DataFrame = df_src.explode("RISKS")

df = df.with_columns(
    pl.when(pl.col("RISKS").is_not_null())
    .then(
        pl.coalesce(
            pl.col("RISKS").struct.field("TOPIC"),
        )
    )
    .otherwise(None)
    .alias("topic")
)

df = df.select(
    pl.col("id"),
    pl.col("topic"),
).drop_nulls(subset=["topic"])

df

id,topic
str,str
"""sz:ds_open-sanctions_NK-25vyVF…","""corp.disqual"""
"""sz:ds_open-sanctions_NK-auyPsL…","""sanction.linked"""
"""sz:ds_open-sanctions_NK-cf4Q3K…","""sanction.linked"""
"""sz:ds_open-sanctions_NK-dNNN56…","""role.rca"""
"""sz:ds_open-sanctions_NK-dNNN56…","""sanction"""
…,…
"""sz:ds_open-sanctions_ru-inn-77…","""sanction.linked"""
"""sz:ds_open-sanctions_rupep-com…","""sanction.linked"""
"""sz:ds_open-sanctions_rupep-com…","""sanction.linked"""


## Extract risk relations

Serialize this dataframe to the `os_risk.csv` CSV file.

In [9]:
df.write_csv(pathlib.Path("os_risk.csv"), separator = ",")

## Parse the Open Ownership dataset

In [10]:
df_src: pl.DataFrame = pl.read_ndjson("data/open-ownership.json")

# normalize IRIs for connecting later
df_src = df_src.replace_column(
    1,
    pl.Series(
        "id",
        map(lambda s: "sz:ds_open-ownership_" + s, df_src["RECORD_ID"])
    ),
)

df_src = df_src.replace_column(
    3,
    pl.Series(
        "class",
        map(lambda s: "sz:" + s.capitalize(), df_src["RECORD_TYPE"])
    ),
)

df_src

DATA_SOURCE,id,statementDate,class,NAMES,PRIMARY_NAME_FULL,personType,ATTRIBUTES,ADDRESSES,IDENTIFIERS,LINKS,RELATIONSHIPS,replaces_statements,REGISTRATION_DATE,dissolutionDate,REGISTRATION_COUNTRY,DATE_OF_BIRTH
str,str,str,str,list[struct[2]],str,str,list[struct[1]],list[struct[3]],list[struct[3]],list[struct[3]],list[struct[7]],list[struct[1]],str,str,str,str
"""OPEN-OWNERSHIP""","""sz:ds_open-ownership_100945215…","""2023-06-18""","""sz:Organization""","[{null,""GOLD WYNN UK HOLDINGS LIMITED""}]",null,null,null,"[{""BUSINESS"",""C/O Fladgate Llp, 16 Great Queen Street, London, WC2B 5DG"",""GB""}]","[{""12524623"",""GB-COH"",""GBR""}]","[{""https://opencorporates.com/companies/gb/12524623"",null,null}, {null,""https://register.openownership.org/entities/18432059995972240708"",null}]","[{null,null,null,null,null,""OOR"",""10094521532396971848""}, {""OOR"",""7584591804488095167"",""shareholding 75% 100%"",""2020-03-18"",""2020-04-29"",null,null}, … {""OOR"",""7584591804488095167"",""appointment_of_board"",""2020-03-18"",""2020-04-29"",null,null}]",null,"""2020-03-18""",null,"""GB""",null
"""OPEN-OWNERSHIP""","""sz:ds_open-ownership_101656327…","""2023-06-18""","""sz:Organization""","[{null,""UPSIDE TECHNOLOGY LIMITED""}]",null,null,null,"[{""BUSINESS"",""Apt 52, 3 Whitehall Court, London, SW1A 2EL"",""GB""}]","[{""12165794"",""GB-COH"",""GBR""}]","[{""https://opencorporates.com/companies/gb/12165794"",null,null}, {null,""https://register.openownership.org/entities/15659422647652524790"",null}]","[{null,null,null,null,null,""OOR"",""10165632722354515453""}, {""OOR"",""598161773989218568"",""shareholding 75% 100%"",""2019-08-20"",null,null,null}, … {""OOR"",""598161773989218568"",""appointment_of_board"",""2019-08-20"",null,null,null}]",null,"""2019-08-20""","""2022-10-11""","""GB""",null
"""OPEN-OWNERSHIP""","""sz:ds_open-ownership_101656327…","""2023-06-18""","""sz:Organization""","[{null,""UPSIDE TECHNOLOGY LIMITED""}]",null,null,null,"[{""BUSINESS"",""Apt 52, 3 Whitehall Court, London, SW1A 2EL"",""GB""}]","[{""12165794"",""GB-COH"",""GBR""}]","[{""https://opencorporates.com/companies/gb/12165794"",null,null}, {null,""https://register.openownership.org/entities/15659422647652524790"",null}]","[{null,null,null,null,null,""OOR"",""10165632722354515453""}, {""OOR"",""598161773989218568"",""shareholding 75% 100%"",""2019-08-20"",null,null,null}, … {""OOR"",""598161773989218568"",""appointment_of_board"",""2019-08-20"",null,null,null}]",null,"""2019-08-20""","""2022-10-11""","""GB""",null
"""OPEN-OWNERSHIP""","""sz:ds_open-ownership_102644597…","""2024-01-03""","""sz:Person""",[],"""Kenneth Kurt Hansen""","""knownPerson""","[{""DK""}]","[{""PRIMARY"",""Finderupvej 61, Kastrup, 2770"",""DK""}]","[{""4008320631"","""",""""}]","[{null,""https://register.openownership.org/entities/4650610720594151415"",null}]","[{null,null,null,null,null,""OOR"",""10264459789712927869""}]","[{""5070572028120081385""}]",null,null,null,null
"""OPEN-OWNERSHIP""","""sz:ds_open-ownership_103690294…","""2023-06-18""","""sz:Person""",[],"""Daniel Simmons""","""knownPerson""","[{""GB""}]","[{""PRIMARY"",""17 St Andrews Crescent, Cardiff, South Glamorgan, CF10 3DB"",""GB""}]",null,"[{null,""https://register.openownership.org/entities/14806142291665148299"",null}]","[{null,null,null,null,null,""OOR"",""10369029484097831758""}]",null,null,null,null,"""1993-02-01"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""OPEN-OWNERSHIP""","""sz:ds_open-ownership_966913957…","""2024-01-03""","""sz:Organization""","[{null,""Ejendomsselskabet Gotland A/S""}, {""A/S KURT HANSEN PROJEKT"",null}, {""A/S KURT HANSEN MASKINFABRIK"",null}]",null,null,null,"[{""BUSINESS"",""Gotlandsvej 5, Svendborg, 5700"",""DK""}]","[{""33050437"",""DK-CVR"",""DNK""}]","[{""https://opencorporates.com/companies/dk/33050437"",null,null}, {null,""https://register.openownership.org/entities/6399142753405592454"",null}]","[{""OOR"",""10264459789712927869"",""shareholding 15%"",""2019-12-19"",null,null,null}, {"

In [11]:
# normalize names from nested attributes
df = df_src.explode("NAMES").explode("ATTRIBUTES")
    
# extract nationality - from person or organization
df = df.with_columns(
    pl.when(pl.col("class") == "sz:Person")
    .then(pl.col("ATTRIBUTES").struct.field("NATIONALITY"))
    .when(pl.col("class") == "sz:Organization")
    .then(pl.col("REGISTRATION_COUNTRY"))
    .otherwise(None)
    .alias("country")
)

# extract name - from person or organization,
# filter records which lack a name
df = df.with_columns(
    pl.when(pl.col("class") == "sz:Person")
    .then(pl.col("PRIMARY_NAME_FULL"))
    .when(pl.col("class") == "sz:Organization")
    .then(pl.col("NAMES").struct.field("PRIMARY_NAME_ORG"))
    .otherwise(None)
    .alias("descrip")
).drop_nulls("descrip")

# normalize addresses from nested attributes
df = df.explode("ADDRESSES")

df = df.with_columns(
    pl.when(pl.col("class") == "sz:Person")
    .then(pl.col("ADDRESSES").struct.field("ADDR_FULL"))
    .when(pl.col("class") == "sz:Organization")
    .then(pl.col("ADDRESSES").struct.field("ADDR_FULL"))
    .otherwise(None)
    .alias("address")
)

# select the results
df = df.select(
    pl.col("id"),
    pl.col("descrip"),
    pl.col("class"),
    pl.col("address"),
    pl.col("country"),
).unique().sort("id")

df

id,descrip,class,address,country
str,str,str,str,str
"""sz:ds_open-ownership_100945215…","""GOLD WYNN UK HOLDINGS LIMITED""","""sz:Organization""","""C/O Fladgate Llp, 16 Great Que…","""GB"""
"""sz:ds_open-ownership_101656327…","""UPSIDE TECHNOLOGY LIMITED""","""sz:Organization""","""Apt 52, 3 Whitehall Court, Lon…","""GB"""
"""sz:ds_open-ownership_102644597…","""Kenneth Kurt Hansen""","""sz:Person""","""Finderupvej 61, Kastrup, 2770""","""DK"""
"""sz:ds_open-ownership_103690294…","""Daniel Simmons""","""sz:Person""","""17 St Andrews Crescent, Cardif…","""GB"""
"""sz:ds_open-ownership_103906995…","""Wyndham James Alexander Plumpt…","""sz:Person""","""Apt 52, 3, Whitehall Court, Lo…","""GB"""
…,…,…,…,…
"""sz:ds_open-ownership_953272446…","""Milla Kier Meraki Hansen""","""sz:Person""","""Valby Langgade 222, 1, Valby, …","""DK"""
"""sz:ds_open-ownership_966913957…","""Ejendomsselskabet Gotland A/S""","""sz:Organization""","""Gotlandsvej 5, Svendborg, 5700""","""DK"""
"""sz:ds_open-ownership_969413650…","""Barry Halstead""","""sz:Person""","""Barry Halstead, Flat 1, 505 Fu…","""GB"""


Serialize this dataframe to the `oo_data.csv` CSV file.

In [12]:
df.write_csv(pathlib.Path("oo_data.csv"), separator = ",")

## Extract UBO relations

Open Ownership describes [_ultimate beneficial ownership_](https://www.beneficialownership.co.uk/) (UBO) details, which provides the "link" category in the generalized anti-fraud data model.

First we'll save the primary keys for the known records in this dataset slice, so that edges are constrained to records that are available here.

In [13]:
ubo_ids: set[ str ] = set(df.select("id").to_series().to_list())

In [14]:
df = df_src.explode("RELATIONSHIPS")

df = df.with_columns([
    pl.col("id").alias("src_id"),
    pl.col("RELATIONSHIPS").struct.field("REL_POINTER_KEY").alias("dst_id"),
    pl.col("RELATIONSHIPS").struct.field("REL_POINTER_ROLE").alias("role"),
    pl.col("RELATIONSHIPS").struct.field("REL_POINTER_FROM_DATE").alias("date"),
]).drop_nulls(subset=["src_id", "dst_id", "role"])

df = df.select(
    pl.col("src_id"),
    pl.col("dst_id"),
    pl.col("role"),
    pl.col("date"),
).sort("src_id", "dst_id")

# normalize IRIs for connecting later
df = df.replace_column(
    1,
    pl.Series(
        "dst_id",
        map(lambda s: "sz:ds_open-ownership_" + s, df["dst_id"])
    ),
)

# only include relationships where both records are available
df = df.filter(
    pl.col("src_id").is_in(ubo_ids) & pl.col("dst_id").is_in(ubo_ids)
)

df

src_id,dst_id,role,date
str,str,str,str
"""sz:ds_open-ownership_100945215…","""sz:ds_open-ownership_758459180…","""shareholding 75% 100%""","""2020-03-18"""
"""sz:ds_open-ownership_100945215…","""sz:ds_open-ownership_758459180…","""appointment_of_board""","""2020-03-18"""
"""sz:ds_open-ownership_100945215…","""sz:ds_open-ownership_758459180…","""voting_rights 75% 100%""","""2020-03-18"""
"""sz:ds_open-ownership_101656327…","""sz:ds_open-ownership_598161773…","""shareholding 75% 100%""","""2019-08-20"""
"""sz:ds_open-ownership_101656327…","""sz:ds_open-ownership_598161773…","""voting_rights 75% 100%""","""2019-08-20"""
…,…,…,…
"""sz:ds_open-ownership_944705691…","""sz:ds_open-ownership_287957459…","""other_influence_or_control""","""2020-04-14"""
"""sz:ds_open-ownership_966913957…","""sz:ds_open-ownership_102644597…","""shareholding 15%""","""2019-12-19"""
"""sz:ds_open-ownership_966913957…","""sz:ds_open-ownership_102644597…","""voting_rights 15%""","""2019-12-19"""


Serialize this dataframe to the `oo_ubos.csv` CSV file.

In [15]:
df.write_csv(pathlib.Path("oo_ubos.csv"), separator = ",")

---